# 12b — specificity addendum

Note: this notebook reruns the existing final model's single test-set evaluation from notebook 12 to derive one additional metric (specificity) that was not computed there. It does not constitute a second, independent look at the test set for model selection — the model and checkpoint are already fixed and unchanged.

In [1]:
import json
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

from citrus_model import CitrusNet
from citrus_common import get_transforms, CitrusLeafDataset

In [2]:
CHECKPOINT_PATH = "checkpoints/baseline_best.pt"
STATS_PATH = "normalization_stats.json"
TEST_ROOT = "processed/224"
RESULTS_PATH = "final_test_results_citrusnet.json"

class_to_idx = {"aphids": 0, "gummosis": 1, "healthy": 2, "leaf_minnor": 3}
class_names = [name for name, _ in sorted(class_to_idx.items(), key=lambda x: x[1])]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Load the finalized CitrusNet checkpoint. No retraining.

In [3]:
model = CitrusNet(num_classes=4)
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

sum(p.numel() for p in model.parameters())

416036

Test set, same transform and stats as notebook 12 (project's own normalization, not ImageNet).

In [4]:
with open(STATS_PATH) as f:
    stats = json.load(f)
mean, std = tuple(stats["mean"]), tuple(stats["std"])

test_transform = get_transforms(augment=False, mean=mean, std=std, size=224)
test_dataset = CitrusLeafDataset(root_dir=TEST_ROOT, split="test", transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

len(test_dataset)

240

Single inference pass over the test set.

In [5]:
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_labels.append(labels.numpy())
        all_preds.append(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

all_labels = np.concatenate(all_labels)
all_preds = np.concatenate(all_preds)
all_probs = np.concatenate(all_probs)

len(all_labels)

240

Accuracy / confusion matrix / classification report — should match notebook 12 exactly (96.67%, 232/240).

In [6]:
test_acc = (all_preds == all_labels).mean()
n_correct = int((all_preds == all_labels).sum())
print(f"test accuracy: {test_acc:.4f} ({n_correct}/{len(all_labels)})")

cm = confusion_matrix(all_labels, all_preds, labels=list(range(4)))
print(cm)

report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=class_names))

test accuracy: 0.9667 (232/240)
[[60  0  0  0]
 [ 0 58  5  2]
 [ 0  0 51  0]
 [ 0  1  0 63]]
              precision    recall  f1-score   support

      aphids       1.00      1.00      1.00        60
    gummosis       0.98      0.89      0.94        65
     healthy       0.91      1.00      0.95        51
 leaf_minnor       0.97      0.98      0.98        64

    accuracy                           0.97       240
   macro avg       0.97      0.97      0.97       240
weighted avg       0.97      0.97      0.97       240



Specificity per class, derived from the confusion matrix (sklearn has no built-in for this).

In [7]:
total = cm.sum()
specificity = {}

for i, name in enumerate(class_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = total - tp - fp - fn
    specificity[name] = tn / (tn + fp)

spec_table = pd.DataFrame({
    "precision": [report[c]["precision"] for c in class_names],
    "recall (sensitivity)": [report[c]["recall"] for c in class_names],
    "specificity": [specificity[c] for c in class_names],
    "f1": [report[c]["f1-score"] for c in class_names],
}, index=class_names)

spec_table

,precision,recall (sensitivity),specificity,f1
aphids,1.000000,1.000000,1.000000,1.000000
gummosis,0.983051,0.892308,0.994286,0.935484
healthy,0.910714,1.000000,0.973545,0.953271
leaf_minnor,0.969231,0.984375,0.988636,0.976744


Per-class one-vs-rest ROC-AUC, same method as notebook 12.

In [8]:
roc_auc = {}
for i, name in enumerate(class_names):
    binary_labels = (all_labels == i).astype(int)
    roc_auc[name] = roc_auc_score(binary_labels, all_probs[:, i])

mean_auc = float(np.mean(list(roc_auc.values())))
roc_auc, mean_auc

({'aphids': 1.0,
  'gummosis': 0.9992967032967033,
  'healthy': 1.0,
  'leaf_minnor': 0.9892578125},
 0.9971386289491758)

Param count, model size, single-image latency — same procedure as notebook 12.

In [9]:
def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

param_count = sum(p.numel() for p in model.parameters())
model_size_mb = param_count * 4 / (1024 ** 2)  # float32

single_image, _ = test_dataset[0]
single_image = single_image.unsqueeze(0).to(device)

with torch.no_grad():
    for _ in range(10):
        model(single_image)
sync()

latencies = []
with torch.no_grad():
    for _ in range(100):
        sync()
        t0 = time.perf_counter()
        model(single_image)
        sync()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)

latencies = np.array(latencies)
mean_latency = float(latencies.mean())
std_latency = float(latencies.std())
throughput = 1000 / mean_latency

param_count, round(model_size_mb, 4), round(mean_latency, 4), round(std_latency, 4), round(throughput, 1)

(416036, 1.5871, 2.7019, 1.0864, 370.1)

Save results in the same schema as notebook 14's per-architecture files, under a new filename. Does not touch final_test_results.json.

In [10]:
results = {
    "architecture": "citrusnet",
    "test_accuracy": float(test_acc),
    "macro_precision": report["macro avg"]["precision"],
    "macro_recall": report["macro avg"]["recall"],
    "macro_f1": report["macro avg"]["f1-score"],
    "weighted_precision": report["weighted avg"]["precision"],
    "weighted_recall": report["weighted avg"]["recall"],
    "weighted_f1": report["weighted avg"]["f1-score"],
    "per_class": {
        name: {
            "precision": report[name]["precision"],
            "recall": report[name]["recall"],
            "sensitivity": report[name]["recall"],
            "specificity": specificity[name],
            "f1": report[name]["f1-score"],
            "roc_auc": roc_auc[name],
        }
        for name in class_names
    },
    "mean_roc_auc": mean_auc,
    "param_count": int(param_count),
    "model_size_mb": model_size_mb,
    "mean_inference_latency_ms": mean_latency,
    "std_inference_latency_ms": std_latency,
    "throughput_images_per_sec": throughput,
}

with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

RESULTS_PATH

'final_test_results_citrusnet.json'

In [11]:
print(f"accuracy {test_acc:.4f} ({n_correct}/{len(all_labels)}) - matches notebook 12's 96.67% (232/240)")
print(f"macro F1 {report['macro avg']['f1-score']:.4f}, mean ROC-AUC {mean_auc:.4f} - consistent with notebook 12")
print()
print("per-class specificity (new metric, not in notebook 12):")
for name in class_names:
    print(f"  {name}: {specificity[name]:.4f}")

accuracy 0.9667 (232/240) - matches notebook 12's 96.67% (232/240)
macro F1 0.9664, mean ROC-AUC 0.9971 - consistent with notebook 12

per-class specificity (new metric, not in notebook 12):
  aphids: 1.0000
  gummosis: 0.9943
  healthy: 0.9735
  leaf_minnor: 0.9886
